# CineTech: Data Preprocessing & Exploratory Data Analysis

**Project:** Hybrid movie recommendation for international and Bollywood cinema  
**Notebook purpose:** document how CineTech loads, cleans, combines, and explores its Kaggle catalogues.

The notebook uses the CSV files already included in `data/kaggle/`. It does not need a TMDB token or network access. Run it from the project root in VS Code or Jupyter. The analysis focuses on data quality and catalog patterns; it does not train or evaluate a recommendation model.

## 1. Import libraries and locate the data

Required libraries: pandas, numpy, matplotlib, and seaborn. The recommendation project's Python requirements already include pandas and numpy; install matplotlib and seaborn in the notebook's selected environment if needed.

In [ ]:
from pathlib import Path
import json
import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)

# Supports running from the project root or from a nested notebooks/ directory.
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in candidates if (p / "data" / "kaggle").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find data/kaggle. Open this notebook from inside the CineTech project.")

DATA_DIR = PROJECT_ROOT / "data" / "kaggle"
print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)

## 2. Load the source files and inspect their shape

The project combines TMDB 5000 movie metadata and credits with a poster catalogue and an Indian movie catalogue. First inspect the raw data before cleaning it.

In [ ]:
movies_raw = pd.read_csv(DATA_DIR / "tmdb_5000_movies.csv")
credits_raw = pd.read_csv(DATA_DIR / "tmdb_5000_credits.csv")
posters_raw = pd.read_csv(DATA_DIR / "mymoviedb_posters.csv")
bollywood_raw = pd.read_csv(DATA_DIR / "timdb_bollywood.csv")

for name, frame in {
    "TMDB movies": movies_raw,
    "TMDB credits": credits_raw,
    "Poster catalogue": posters_raw,
    "Indian/Bollywood catalogue": bollywood_raw,
}.items():
    print(f"{name:28} rows={len(frame):5,} columns={len(frame.columns):2}")
    display(frame.head(3))

## 3. Check missing values and duplicate records

This gives a quick data-quality overview. Missing values are not always errors: for example, some films have no overview or poster in the source data.

In [ ]:
def quality_report(frame, label):
    report = pd.DataFrame({
        "missing_count": frame.isna().sum(),
        "missing_percent": (frame.isna().mean() * 100).round(2),
        "dtype": frame.dtypes.astype(str),
    }).sort_values("missing_percent", ascending=False)
    print(f"{label}: {len(frame):,} rows; exact duplicate rows={frame.duplicated().sum():,}")
    display(report.head(15))

quality_report(movies_raw, "TMDB movies")
quality_report(bollywood_raw, "Bollywood catalogue")
print("Duplicate TMDB IDs:", movies_raw["id"].duplicated().sum())
print("Duplicate TMDB credit IDs:", credits_raw["movie_id"].duplicated().sum())

## 4. Preprocess TMDB metadata and credits

Parse the JSON-like genre and cast columns, convert numeric/date fields, remove unusable titles and duplicate IDs, and attach a cast count from the credits file. Keep a small, consistent set of columns for analysis.

In [ ]:
def parse_json_list(value):
    if pd.isna(value) or not str(value).strip():
        return []
    try:
        parsed = json.loads(value)
        return parsed if isinstance(parsed, list) else []
    except (json.JSONDecodeError, TypeError):
        return []

tmdb = movies_raw.copy()
tmdb["title"] = tmdb["title"].fillna(tmdb["original_title"])
tmdb["release_date"] = pd.to_datetime(tmdb["release_date"], errors="coerce")
tmdb["year"] = tmdb["release_date"].dt.year
for column in ["vote_average", "vote_count", "popularity", "runtime", "budget", "revenue"]:
    tmdb[column] = pd.to_numeric(tmdb[column], errors="coerce")

tmdb["genres_list"] = tmdb["genres"].apply(parse_json_list)
tmdb["genres_text"] = tmdb["genres_list"].apply(lambda items: " | ".join(
    item.get("name", "") for item in items if isinstance(item, dict) and item.get("name")
))
tmdb["genre_count"] = tmdb["genres_list"].apply(len)

credits = credits_raw.drop_duplicates("movie_id").copy()
credits["cast_list"] = credits["cast"].apply(parse_json_list)
credits["cast_count"] = credits["cast_list"].apply(len)
tmdb = tmdb.merge(credits[["movie_id", "cast_count"]], left_on="id", right_on="movie_id", how="left")
tmdb["cast_count"] = tmdb["cast_count"].fillna(0).astype(int)
tmdb = tmdb.dropna(subset=["title"]).drop_duplicates("id").reset_index(drop=True)

tmdb_clean = tmdb[[
    "id", "title", "original_title", "year", "release_date", "original_language",
    "genres_text", "genre_count", "overview", "vote_average", "vote_count",
    "popularity", "runtime", "budget", "revenue", "cast_count"
]].copy()
tmdb_clean["industry"] = "International"
tmdb_clean["rating_source"] = "TMDB"
print(f"Clean TMDB catalogue: {len(tmdb_clean):,} films")
display(tmdb_clean.head())

## 5. Match poster URLs to TMDB titles

Poster matching uses normalized title and release year when possible. A title-only match is used as a fallback, mirroring the app's local catalogue logic. A missing poster is kept as a missing value rather than dropping the film.

In [ ]:
def normalize_title(value):
    value = unicodedata.normalize("NFKD", str(value or ""))
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    return re.sub(r"[^a-z0-9]+", " ", value.lower()).strip()

posters = posters_raw.copy()
posters["title_key"] = posters["Title"].map(normalize_title)
posters["year"] = pd.to_datetime(posters["Release_Date"], errors="coerce").dt.year
posters["Poster_Url"] = posters["Poster_Url"].where(
    posters["Poster_Url"].astype(str).str.match(r"^https://", case=False, na=False)
)

poster_title_year = (
    posters.dropna(subset=["Poster_Url"])
    .drop_duplicates(["title_key", "year"])
    .set_index(["title_key", "year"])["Poster_Url"]
)
poster_title = (
    posters.dropna(subset=["Poster_Url"])
    .drop_duplicates("title_key")
    .set_index("title_key")["Poster_Url"]
)

tmdb_clean["title_key"] = tmdb_clean["title"].map(normalize_title)
tmdb_clean["poster_url"] = [
    poster_title_year.get((key, year), poster_title.get(key, np.nan))
    for key, year in zip(tmdb_clean["title_key"], tmdb_clean["year"])
]
print("TMDB poster coverage:", f"{tmdb_clean['poster_url'].notna().mean():.1%}")
display(tmdb_clean[["title", "year", "poster_url"]].head(10))

## 6. Preprocess the Bollywood catalogue and combine the films

The Indian catalogue has alternate title columns and IMDb ratings. Normalize the title/year keys to avoid adding a film already present in the international catalogue. This is a metadata merge for discovery; it does not imply that every title has complete metadata or a poster.

In [ ]:
bolly = bollywood_raw.copy()
bolly["title"] = (
    bolly["title_y"].fillna("").astype(str).str.strip()
    .replace("", np.nan)
    .fillna(bolly["title_x"])
    .fillna(bolly["original_title"])
)
bolly["year"] = pd.to_numeric(bolly["year_of_release"], errors="coerce")
bolly["rating"] = pd.to_numeric(bolly["imdb_rating"], errors="coerce")
bolly["vote_count"] = pd.to_numeric(bolly["imdb_votes"], errors="coerce")
bolly["runtime"] = pd.to_numeric(bolly["runtime"], errors="coerce")
bolly["genres_text"] = bolly["genres"].fillna("").astype(str).str.replace("|", " | ", regex=False)
bolly["overview"] = bolly["story"].fillna(bolly["summary"]).fillna("")
bolly["poster_url"] = bolly["poster_path"]
bolly["title_key"] = bolly["title"].map(normalize_title)
bolly["industry"] = "Bollywood"
bolly["rating_source"] = "IMDb"
bolly["id"] = "bollywood-" + bolly.index.astype(str)

existing_keys = set(zip(tmdb_clean["title_key"], tmdb_clean["year"]))
bolly_keys = list(zip(bolly["title_key"], bolly["year"]))
bolly = bolly.loc[
    bolly["title"].notna()
    & ~pd.Series(bolly_keys, index=bolly.index).isin(existing_keys)
].copy()

tmdb_for_merge = tmdb_clean.assign(rating=tmdb_clean["vote_average"])
keep = ["id", "title", "year", "industry", "rating", "rating_source",
        "vote_count", "genres_text", "overview", "runtime", "poster_url"]
combined = pd.concat([
    tmdb_for_merge[keep],
    bolly[keep],
], ignore_index=True)
combined["year"] = pd.to_numeric(combined["year"], errors="coerce").astype("Int64")
combined["rating"] = pd.to_numeric(combined["rating"], errors="coerce")
combined["vote_count"] = pd.to_numeric(combined["vote_count"], errors="coerce")
combined["runtime"] = pd.to_numeric(combined["runtime"], errors="coerce")

print(f"Combined catalogue: {len(combined):,} films")
display(combined.groupby("industry").agg(
    films=("title", "count"),
    years=("year", "nunique"),
    poster_coverage=("poster_url", lambda s: s.notna().mean()),
    average_rating=("rating", "mean"),
).round(2))

## 7. Exploratory data analysis (EDA)

### Release-year coverage and film ratings

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
year_counts = combined.dropna(subset=["year"]).groupby("year").size()
axes[0].plot(year_counts.index.astype(int), year_counts.values, color="#1677b8")
axes[0].set(title="Films by release year", xlabel="Release year", ylabel="Number of films")

sns.histplot(data=combined, x="rating", hue="industry", bins=20, element="step",
             stat="count", common_norm=False, ax=axes[1])
axes[1].set(title="Source audience ratings", xlabel="Rating", ylabel="Number of films")
plt.tight_layout()
plt.show()

### Genre distribution and poster coverage

In [ ]:
genre_rows = (
    combined[["industry", "genres_text"]]
    .dropna(subset=["genres_text"])
    .assign(genre=lambda d: d["genres_text"].str.split(r"\s*\|\s*"))
    .explode("genre")
)
genre_rows["genre"] = genre_rows["genre"].str.strip()
genre_rows = genre_rows[genre_rows["genre"].ne("")]

top_genres = genre_rows["genre"].value_counts().head(12).sort_values()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
top_genres.plot(kind="barh", ax=axes[0], color="#1677b8")
axes[0].set(title="Most common genres (combined catalogue)", xlabel="Film count", ylabel="Genre")

poster_rates = combined.groupby("industry")["poster_url"].apply(lambda s: s.notna().mean()).sort_values()
poster_rates.plot(kind="barh", ax=axes[1], color="#18a999", xlim=(0, 1))
axes[1].set(title="Poster URL coverage by industry", xlabel="Share with poster URL", ylabel="")
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
plt.tight_layout()
plt.show()

## 8. Summary and limitations

The outputs above describe the catalog's coverage, source ratings, genres, and poster availability. Ratings from TMDB and IMDb come from different sources and should be shown with their source labels. Missing poster URLs are expected for some entries. The catalogue is a static snapshot, so it does not provide current releases or live streaming availability.

### Suggested project-report summary

> The datasets were loaded and inspected for missing values and duplicates. TMDB genre and cast fields were parsed from JSON strings, date and rating fields were converted to suitable types, and credits were joined to movie metadata. Poster links were matched by normalized title and year where possible. Bollywood records were cleaned and combined with the international catalogue after title/year duplicate checks. EDA charts summarize release years, genre frequency, source ratings, and poster URL coverage. This preprocessing prepares a consistent movie catalogue for CineTech's survey-based hybrid recommender.